In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from sklearn.model_selection import train_test_split
import random
import numpy as np

# Set random seeds cho reproducibility
random.seed(42)
np.random.seed(42)

print("✓ Libraries imported successfully")


In [ ]:
def convert_voc_to_yolo(xml_file):
    """
    Convert Pascal VOC XML annotation to YOLO format.
    
    Args:
        xml_file: Path to XML annotation file
        
    Returns:
        list: List of YOLO format annotations [class_id, x_center, y_center, width, height]
    """
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # Get image size
    size = root.find('size')
    img_width = int(size.find('width').text)
    img_height = int(size.find('height').text)
    
    yolo_annotations = []
    
    # Process each object
    for obj in root.findall('object'):
        class_name = obj.find('name').text
        
        # Class mapping (pothole = 0)
        class_id = 0
        
        # Get bounding box
        bbox = obj.find('bndbox')
        xmin = float(bbox.find('xmin').text)
        ymin = float(bbox.find('ymin').text)
        xmax = float(bbox.find('xmax').text)
        ymax = float(bbox.find('ymax').text)
        
        # Convert to YOLO format (normalized x_center, y_center, width, height)
        x_center = ((xmin + xmax) / 2) / img_width
        y_center = ((ymin + ymax) / 2) / img_height
        width = (xmax - xmin) / img_width
        height = (ymax - ymin) / img_height
        
        yolo_annotations.append([class_id, x_center, y_center, width, height])
    
    return yolo_annotations

# Test the conversion function
test_xml = '/kaggle/input/data-pothole/data/annotations/potholes116.xml'
test_result = convert_voc_to_yolo(test_xml)
print(f"✓ Conversion function tested successfully")
print(f"Sample XML file has {len(test_result)} objects")
print(f"First annotation (YOLO format): {test_result[0]}")


In [ ]:
# Get all XML files
annotations_dir = Path('/kaggle/input/daaaaa/data/annotations')
images_dir = Path('/kaggle/input/daaaaa/data/images')

xml_files = sorted(list(annotations_dir.glob('*.xml')))
print(f"Found {len(xml_files)} XML annotation files")

# Verify corresponding images exist
image_files = []
for xml_file in xml_files:
    img_name = xml_file.stem + '.png'
    img_path = images_dir / img_name
    if img_path.exists():
        image_files.append(img_path)
    else:
        print(f"Warning: Image not found for {xml_file.name}")

print(f"✓ Found {len(image_files)} corresponding images")
print(f"✓ All {len(xml_files)} annotations matched with images")


In [ ]:
# Prepare data for splitting
all_files = [(xml_file, image_files[i]) for i, xml_file in enumerate(xml_files)]

# Split: 70% train, 20% val, 10% test
train_files, temp_files = train_test_split(all_files, test_size=0.30, random_state=42)
val_files, test_files = train_test_split(temp_files, test_size=0.33333, random_state=42)  # 0.33 of 30% = 10%

print(f"Dataset split:")
print(f"  Train: {len(train_files)} images ({len(train_files)/len(all_files)*100:.1f}%)")
print(f"  Val:   {len(val_files)} images ({len(val_files)/len(all_files)*100:.1f}%)")
print(f"  Test:  {len(test_files)} images ({len(test_files)/len(all_files)*100:.1f}%)")
print(f"  Total: {len(all_files)} images")


In [ ]:
# Create YOLO directory structure
dataset_root = Path('dataset')

# Remove old dataset if exists
if dataset_root.exists():
    shutil.rmtree(dataset_root)
    print("Removed old dataset directory")

# Create directories
for split in ['train', 'val', 'test']:
    (dataset_root / split / 'images').mkdir(parents=True, exist_ok=True)
    (dataset_root / split / 'labels').mkdir(parents=True, exist_ok=True)

print("✓ Created YOLO directory structure:")
print("  dataset/")
print("    train/images/")
print("    train/labels/")
print("    val/images/")
print("    val/labels/")
print("    test/images/")
print("    test/labels/")


In [ ]:
def process_and_save_data(file_pairs, split_name):
    """
    Convert annotations and copy files to YOLO format structure.
    
    Args:
        file_pairs: List of (xml_file, img_file) tuples
        split_name: 'train', 'val', or 'test'
    """
    print(f"\nProcessing {split_name} set...")
    
    for xml_file, img_file in file_pairs:
        # Convert annotation
        yolo_annotations = convert_voc_to_yolo(xml_file)
        
        # Get base filename
        base_name = xml_file.stem
        
        # Copy image
        dst_img = dataset_root / split_name / 'images' / f'{base_name}.png'
        shutil.copy2(img_file, dst_img)
        
        # Save YOLO format label
        dst_label = dataset_root / split_name / 'labels' / f'{base_name}.txt'
        with open(dst_label, 'w') as f:
            for ann in yolo_annotations:
                # Format: class_id x_center y_center width height
                f.write(f"{ann[0]} {ann[1]:.6f} {ann[2]:.6f} {ann[3]:.6f} {ann[4]:.6f}\n")
    
    print(f"✓ {split_name}: Processed {len(file_pairs)} images and annotations")

# Process all splits
process_and_save_data(train_files, 'train')
process_and_save_data(val_files, 'val')
process_and_save_data(test_files, 'test')

print("\n✓ Dataset conversion completed successfully!")


In [ ]:
# Create data.yaml file
data_yaml_content = """# YOLOv8 Pothole Detection Dataset Configuration

# Dataset paths (relative to this file)
path: ./dataset
train: train/images
val: val/images
test: test/images

# Number of classes
nc: 1

# Class names
names:
  0: pothole
"""

with open('data.yaml', 'w') as f:
    f.write(data_yaml_content)

print("✓ Created data.yaml")
print("\nContent:")
print(data_yaml_content)


In [ ]:
# Install ultralytics
!pip install -q ultralytics>=8.0.0

print("✓ Ultralytics installed successfully")


In [ ]:
# Check GPU availability
import torch
from ultralytics import YOLO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")
else:
    print("⚠ Warning: CUDA not available, training will be slow on CPU")


In [ ]:
# Load YOLOv8m pretrained model
model = YOLO('yolov8m.pt')

print("✓ YOLOv8m model loaded successfully")
print(f"Model: {model.model_name if hasattr(model, 'model_name') else 'YOLOv8m'}")


In [ ]:
# Train model với hyperparameters được tối ưu hóa cho độ chính xác cao
print("Starting training...")
print("="*70)

results = model.train(
    # Dataset configuration
    data='/kaggle/working/data.yaml',
    
    # Training parameters
    epochs=100,
    patience=20,          # Early stopping patience
    batch=16,            # Batch size (phù hợp cho T4 X2)
    imgsz=640,           # Input image size
    
    # Optimizer settings
    optimizer='AdamW',   # AdamW optimizer
    lr0=0.001,           # Initial learning rate
    lrf=0.01,            # Final learning rate factor (final_lr = lr0 * lrf)
    momentum=0.937,      # SGD momentum/Adam beta1
    weight_decay=0.0005, # Weight decay
    
    # Data Augmentation (mạnh cho dataset nhỏ)
    hsv_h=0.015,         # HSV-Hue augmentation
    hsv_s=0.7,           # HSV-Saturation augmentation (quan trọng cho màu sắc ổ gà)
    hsv_v=0.4,           # HSV-Value/Brightness augmentation (điều kiện ánh sáng)
    degrees=10,          # Rotation augmentation (±deg)
    translate=0.1,       # Translation augmentation (±fraction)
    scale=0.5,           # Scaling augmentation (±gain)
    shear=5,             # Shear augmentation (±deg)
    perspective=0.0001,  # Perspective augmentation
    flipud=0.0,          # Vertical flip probability (ổ gà luôn ở dưới)
    fliplr=0.5,          # Horizontal flip probability
    mosaic=1.0,          # Mosaic augmentation probability
    mixup=0.1,           # Mixup augmentation probability
    
    # Additional settings
    copy_paste=0.0,      # Copy-paste augmentation probability
    auto_augment='randaugment',  # Auto augmentation policy
    
    # Validation and saving
    save=True,           # Save checkpoints
    save_period=-1,      # Save checkpoint every x epochs (-1 = only save last and best)
    val=True,            # Validate during training
    plots=True,          # Generate plots
    
    # Device and workers
    device=0,            # GPU device (0 for first GPU, or 'cpu')
    workers=8,           # Number of worker threads for data loading
    
    # Project settings
    project='pothole_detection',
    name='yolov8m_run',
    exist_ok=True,
    
    # Advanced settings
    pretrained=True,     # Use pretrained weights
    verbose=True,        # Verbose output
    seed=42,             # Random seed for reproducibility
    deterministic=True,  # Deterministic mode
    
    # NMS settings (for validation)
    conf=0.25,           # Confidence threshold
    iou=0.7,             # IoU threshold for NMS
)

print("="*70)
print("✓ Training completed!")


In [ ]:
# Display training results
print("\n" + "="*70)
print("TRAINING RESULTS SUMMARY")
print("="*70)

# Get best model path
best_model_path = 'pothole_detection/yolov8m_run/weights/best.pt'
last_model_path = 'pothole_detection/yolov8m_run/weights/last.pt'

print(f"\n✓ Best model saved at: {best_model_path}")
print(f"✓ Last model saved at: {last_model_path}")

# Print key metrics
if hasattr(results, 'results_dict'):
    metrics = results.results_dict
    print("\nKey Metrics:")
    for key, value in metrics.items():
        print(f"  {key}: {value}")
        
print("\n✓ Check 'pothole_detection/yolov8m_run/' for detailed results and plots")


In [ ]:
# Load best model
best_model = YOLO('pothole_detection/yolov8m_run/weights/best.pt')

print("✓ Loaded best model for evaluation")


In [ ]:
# Evaluate on validation set
print("\n" + "="*70)
print("VALIDATION SET EVALUATION")
print("="*70)

val_metrics = best_model.val(
    data='data.yaml',
    split='val',
    batch=16,
    imgsz=640,
    conf=0.25,
    iou=0.7,
    plots=True,
    save_json=True,
    project='pothole_detection',
    name='val_results',
    exist_ok=True
)

print("\nValidation Metrics:")
print(f"  mAP@0.5: {val_metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {val_metrics.box.map:.4f}")
print(f"  Precision: {val_metrics.box.mp:.4f}")
print(f"  Recall: {val_metrics.box.mr:.4f}")

# Check if targets are met
print("\nTarget Achievement:")
print(f"  mAP@0.5 > 0.85: {'✓ PASSED' if val_metrics.box.map50 > 0.85 else '✗ NOT MET'}")
print(f"  mAP@0.5:0.95 > 0.65: {'✓ PASSED' if val_metrics.box.map > 0.65 else '✗ NOT MET'}")
print(f"  Precision > 0.80: {'✓ PASSED' if val_metrics.box.mp > 0.80 else '✗ NOT MET'}")
print(f"  Recall > 0.80: {'✓ PASSED' if val_metrics.box.mr > 0.80 else '✗ NOT MET'}")


In [ ]:
# Evaluate on test set
print("\n" + "="*70)
print("TEST SET EVALUATION")
print("="*70)

test_metrics = best_model.val(
    data='data.yaml',
    split='test',
    batch=16,
    imgsz=640,
    conf=0.25,
    iou=0.7,
    plots=True,
    save_json=True,
    project='pothole_detection',
    name='test_results',
    exist_ok=True
)

print("\nTest Metrics:")
print(f"  mAP@0.5: {test_metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {test_metrics.box.map:.4f}")
print(f"  Precision: {test_metrics.box.mp:.4f}")
print(f"  Recall: {test_metrics.box.mr:.4f}")

print("\n✓ Test evaluation completed")


In [ ]:
# Visualize predictions on test set
print("\n" + "="*70)
print("GENERATING PREDICTIONS ON TEST SET")
print("="*70)

predict_results = best_model.predict(
    source='dataset/test/images',
    save=True,
    save_txt=True,
    save_conf=True,
    conf=0.25,
    iou=0.7,
    imgsz=640,
    project='pothole_detection',
    name='predictions',
    exist_ok=True,
    line_width=2,
    show_labels=True,
    show_conf=True
)

print(f"\n✓ Predictions saved to: pothole_detection/predictions/")
print(f"✓ Generated predictions for {len(predict_results)} test images")


In [ ]:
# Export to ONNX format (cross-platform deployment)
print("\n" + "="*70)
print("EXPORTING MODEL TO ONNX")
print("="*70)

try:
    onnx_path = best_model.export(
        format='onnx',
        imgsz=640,
        optimize=True,
        simplify=True,
        dynamic=False,
        opset=12
    )
    print(f"✓ Model exported to ONNX: {onnx_path}")
except Exception as e:
    print(f"✗ ONNX export failed: {e}")


In [ ]:
# Export to TensorRT format (NVIDIA GPU deployment - faster inference)
print("\n" + "="*70)
print("EXPORTING MODEL TO TENSORRT")
print("="*70)

try:
    # TensorRT export requires CUDA and TensorRT installed
    tensorrt_path = best_model.export(
        format='engine',
        imgsz=640,
        half=True,  # FP16 precision for faster inference
        device=0
    )
    print(f"✓ Model exported to TensorRT: {tensorrt_path}")
except Exception as e:
    print(f"⚠ TensorRT export skipped: {e}")
    print("Note: TensorRT requires CUDA and TensorRT to be installed")


In [ ]:
# Display sample predictions with matplotlib
try:
    import matplotlib.pyplot as plt
    import cv2
    from pathlib import Path
    
    # Get some test images
    test_img_dir = Path('dataset/test/images')
    test_images = list(test_img_dir.glob('*.png'))[:6]  # Get first 6 images
    
    if len(test_images) > 0:
        print(f"Displaying {len(test_images)} sample predictions...")
        
        # Create predictions
        results = best_model.predict(
            source=test_images,
            conf=0.25,
            iou=0.7,
            save=False,
            verbose=False
        )
        
        # Plot results
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, (result, img_path) in enumerate(zip(results, test_images)):
            if idx >= 6:
                break
            
            # Get annotated image
            img_with_boxes = result.plot()
            
            # Convert BGR to RGB for matplotlib
            img_rgb = cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB)
            
            # Display
            axes[idx].imshow(img_rgb)
            axes[idx].axis('off')
            axes[idx].set_title(f'{img_path.name}\nDetections: {len(result.boxes)}')
        
        plt.tight_layout()
        plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print(f"✓ Sample predictions saved to: sample_predictions.png")
    else:
        print("⚠ No test images found")
        
except Exception as e:
    print(f"⚠ Visualization skipped: {e}")
    print("Tip: Run the training cells first to generate test set")
